# Notebook 03 - Random Forest baseline (Emily)

Track A, Topic 4: Intergenerational Mobility Across US Counties.
This notebook is the RF half of the OLS-vs-RF comparison; the OLS baseline lives in
`02_ols_baseline.ipynb` (Kabir).

## Project context

We predict the Opportunity Atlas county-level outcome `kfr_pooled_pooled_p25` (mean
income rank at parent p25) from Chetty et al. (2014) Online Data Table 8 covariates
broadcast onto the county frame via the county-to-CZ crosswalk. Paul's pipeline has
already cleaned and saved the merged frame as `data/processed/features.parquet`
(3,130 counties x 106 columns, 1.78 MB) so this notebook is read-only on the data
side.

The notebook produces three artefacts:

1. **Result 3 - mean-decrease-in-impurity (MDI) ranking** of all 38 CZ covariates
   from a single random forest fit, saved as `report/result3_feature_importance.png`.
2. **OLS-aligned permutation importance** on the same six predictors that enter
   Kabir's OLS spec, saved as
   `report/result3b_permutation_importance.png` - this lets the report compare the
   RF feature ranking directly against the OLS coefficients.
3. **Partial dependence plots** for the top four MDI predictors, saved as
   `report/result3c_partial_dependence.png` - this is what surfaces the nonlinear
   structure that the OLS spec assumes away.

All RF fits use `RandomForestRegressor(n_estimators=300, random_state=148)` and an
80/20 train/test split with the same seed for reproducibility.

## Imports and path setup

Pull the scientific-Python stack plus the project's own `src.modeling` module
(eight `<= 10`-line helpers wrapping sklearn calls). The path-discovery loop walks up
from the notebook's working directory to find the project root.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'src' / 'modeling.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
ROOT = next(c for c in [_here, *_here.parents] if (c / 'src' / 'modeling.py').exists())

from src.modeling import (
    CZ_COVARIATES,
    OLS_PREDICTORS,
    OUTCOME,
    evaluate,
    importance_series,
    perm_importance_series,
    prepare_xy,
    save_horizontal_bars,
    save_pdp_panel,
    split_and_fit,
)

print(f'project root: {ROOT}')
print(f'CZ covariates: {len(CZ_COVARIATES)}   OLS predictors: {len(OLS_PREDICTORS)}')

## 1. Load `features.parquet` and prepare X, y

`prepare_xy` reads the parquet, drops the four rows with missing
`kfr_pooled_pooled_p25` (0.13 percent of the cleaned 3,130 frame), and mean-imputes
the predictor columns. Mean imputation is appropriate here because the only
predictors with non-zero missingness are the income-adjusted test-score percentile
(44 missing) and the Social Capital Index (25 missing) - both already centred at
zero, so the imputed values do not bias the tree splits.

In [ ]:
PARQUET = ROOT / 'data' / 'processed' / 'features.parquet'

X_full, y = prepare_xy(PARQUET, CZ_COVARIATES)
print(f'X_full: {X_full.shape}   y: {y.shape}   outcome: {OUTCOME}')
print(f'y mean: {y.mean():.4f}   y std: {y.std():.4f}')

## 2. Train/test split and fit the all-38-covariate RF

`split_and_fit` runs an 80/20 split with `random_state=148` and fits
`RandomForestRegressor(n_estimators=300, random_state=148, n_jobs=-1)`. The seed is
148 for both the split and the forest so the chart and metrics are bit-for-bit
reproducible across reruns.

In [ ]:
rf_full, X_tr, X_te, y_tr, y_te = split_and_fit(X_full, y)
r2_full, rmse_full = evaluate(rf_full, X_te, y_te)
print(f'X_train: {X_tr.shape}   X_test: {X_te.shape}')
print(f'test R^2:  {r2_full:.4f}')
print(f'test RMSE: {rmse_full:.4f}')

## 3. Result 3 - mean-decrease-in-impurity ranking of all 38 covariates

For each tree node, MDI accumulates the reduction in mean-squared error attributable
to that feature, averaged across all 300 trees. The ranking is fast and exact for
the fitted forest but is biased toward high-cardinality features (continuous
variables can produce more splits than discrete ones) - we cross-check the top
ranks below using permutation importance, which is unbiased in that sense.

In [ ]:
mdi = importance_series(rf_full, CZ_COVARIATES)

print('Top 10 features by mean decrease in impurity:')
for i, (name, val) in enumerate(mdi.head(10).items(), 1):
    print(f'  {i:2d}. {val:.4f}  {name}')

mdi_chart = save_horizontal_bars(
    mdi,
    ROOT / 'report' / 'result3_feature_importance.png',
    title=f'Result 3: RF feature importance (test R^2 = {r2_full:.3f})\n38 CZ-level covariates predicting kfr_pooled_pooled_p25',
    xlabel='Mean decrease in impurity',
)
Image(str(mdi_chart))

## 4. OLS-aligned permutation importance

To compare the RF ranking directly against Kabir's OLS coefficients, we refit the
forest on the same six predictors that enter the OLS spec and compute permutation
importance on the held-out 20 percent. Permutation importance shuffles one column
at a time on the test set and records the resulting drop in test-R^2: the result is
a model-agnostic, scale-free measure of how much each feature actually contributes
to held-out predictive accuracy. We use `n_repeats=20` to stabilise the means.

The smaller R^2 here (relative to the all-38 fit) is expected: the OLS spec
deliberately uses only six features, so the forest is comparing apples to apples
when we eventually contrast its rankings with Kabir's coefficients.

In [ ]:
X6, y6 = prepare_xy(PARQUET, OLS_PREDICTORS)
rf6, X6_tr, X6_te, y6_tr, y6_te = split_and_fit(X6, y6)
r2_six, rmse_six = evaluate(rf6, X6_te, y6_te)
print(f'Six-variable RF: test R^2 = {r2_six:.4f}, RMSE = {rmse_six:.4f}')
print()

perm = perm_importance_series(rf6, X6_te, y6_te, n_repeats=20)
print('Permutation importance on the OLS-aligned six predictors (mean drop in R^2):')
for name, val in perm.items():
    print(f'  {val:+.4f}  {name}')

perm_chart = save_horizontal_bars(
    perm,
    ROOT / 'report' / 'result3b_permutation_importance.png',
    title=f'Permutation importance, six OLS-aligned predictors (test R^2 = {r2_six:.3f})',
    xlabel='Mean decrease in held-out R^2 (n_repeats=20)',
)
Image(str(perm_chart))

## 5. Partial dependence for the top four MDI predictors

PDP plots show the marginal effect of each feature on the predicted outcome,
holding all other features at their average. They surface the nonlinear / threshold
structure the random forest has learned that an OLS line cannot represent. We use
the four highest MDI features from step 3 because those are the ones the forest
relies on most.

In [ ]:
top4 = mdi.head(4).index.tolist()
print('Top 4 features for PDP:', top4)

pdp_chart = save_pdp_panel(
    rf_full,
    X_full,
    top4,
    ROOT / 'report' / 'result3c_partial_dependence.png',
)
Image(str(pdp_chart))

## 6. Progress Report summary

Two-to-three-sentence summary regenerated from live values - drop into the Progress
Report or paste into Slack as-is.

In [ ]:
top3 = ', '.join(mdi.head(3).index.tolist())
perm_top = perm.index[0]

summary = (
    f"On the cleaned {len(X_full)}-row county frame I fit "
    f"RandomForestRegressor(n_estimators=300, random_state=148) on the 38 CZ-level "
    f"covariates with an 80/20 split, getting a test R^2 of {r2_full:.3f} and RMSE "
    f"of {rmse_full:.3f}; the top three predictors by mean decrease in impurity are "
    f"{top3}. "
    f"Refitting on the same six predictors that enter Kabir's OLS spec and using "
    f"permutation importance (which is unbiased toward high-cardinality features) "
    f"the held-out RF reaches R^2 = {r2_six:.3f} and ranks '{perm_top}' first - the "
    f"ranking is consistent with the all-38 result. "
    f"Partial dependence plots for the top four MDI predictors (saved as "
    f"report/result3c_partial_dependence.png) are visibly nonlinear, which is the "
    f"specific structure the linear OLS spec assumes away."
)
print(summary)